<a href="https://colab.research.google.com/github/cwintae/Data-Analysis-/blob/main/Market_Basket_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#impor packages

# pip install pandas
# pip install numpy
# pip install datetime
import pandas as pd
import numpy as np
import datetime as dt

In [3]:
#impor data csv
df = pd.read_csv('Online Retail Data.csv', header=0)
df

,order_id,product_code,product_name,quantity,order_date,price,customer_id
0,493410,TEST001,This is a test product.,5.0,2010-01-04 09:24:00,4.50,12346.0
1,C493411,21539,RETRO SPOTS BUTTER DISH,-1.0,2010-01-04 09:43:00,4.25,14590.0
2,493412,TEST001,This is a test product.,5.0,2010-01-04 09:53:00,4.50,12346.0
3,493413,21724,PANDA AND BUNNIES STICKER SHEET,1.0,2010-01-04 09:54:00,0.85,NaN
4,493413,84578,ELEPHANT TOY WITH BLUE T-SHIRT,1.0,2010-01-04 09:54:00,3.75,NaN
...,...,...,...,...,...,...,...
270539,522423,20969,RED FLORAL FELTCRAFT SHOULDER BAG,4.0,2010-09-14 12:55:00,3.75,13880.0
270540,522426,22631,CIRCUS PARADE LUNCH BOX,2.0,2010-09-14 13:16:00,1.95,17861.0
270541,522426,22629,SPACEBOY LUNCH BOX,2.0,2010-09-14 13:16:00,1.95,17861.0
270542,522426,22659,LUNCH BOX I LOVE LONDON,2.0,2010-09-14 13:16:00,1.95,17861.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270544 entries, 0 to 270543
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      270544 non-null  object 
 1   product_code  270544 non-null  object 
 2   product_name  268524 non-null  object 
 3   quantity      270543 non-null  float64
 4   order_date    270543 non-null  object 
 5   price         270543 non-null  float64
 6   customer_id   213624 non-null  float64
dtypes: float64(3), object(4)
memory usage: 14.4+ MB


# **DATA CLEANSING**

In [6]:
df_clean = df.copy()
# membuat kolom date
df_clean['date'] = pd.to_datetime(df_clean['order_date']).dt.normalize()
# menghapus semua baris tanpa customer_id
df_clean = df_clean[~df_clean['customer_id'].isna()]
# mengkonversi customer_id menjadi string
df_clean['customer_id'] = df_clean['customer_id'].astype(str)
# menghapus semua baris tanpa product_name
df_clean = df_clean[~df_clean['product_name'].isna()]
# membuat semua product_name berhuruf kecil
df_clean['product_name'] = df_clean['product_name'].str.lower()
# menghapus semua baris dengan product_code atau product_name test
df_clean = df_clean[(~df_clean['product_code'].str.lower().str.contains('test')) |
                    (~df_clean['product_name'].str.contains('test '))]
# menghapus baris dengan status cancelled, yaitu yang order_id-nya diawali 'C'
df_clean = df_clean[df_clean['order_id'].str[:1]!='C']
# mengubah nilai quantity yang negatif menjadi positif karena nilai negatif tersebut hanya menandakan order tersebut cancelled
df_clean['quantity'] = df_clean['quantity'].abs()
# menghapus baris dengan price bernilai negatif
df_clean = df_clean[df_clean['price']>0]
# membuat nilai amount, yaitu perkalian antara quantity dan price
df_clean['amount'] = df_clean['quantity'] * df_clean['price']
# mengganti product_name dari product_code yang memiliki beberapa product_name dengan salah satu product_name-nya yang paling sering muncul
most_freq_product_name = df_clean.groupby(['product_code','product_name'], as_index=False).agg(order_cnt=('order_id','nunique')).sort_values(['product_code','order_cnt'], ascending=[True,False])
most_freq_product_name['rank'] = most_freq_product_name.groupby('product_code')['order_cnt'].rank(method='first', ascending=False)
most_freq_product_name = most_freq_product_name[most_freq_product_name['rank']==1].drop(columns=['order_cnt','rank'])
df_clean = df_clean.merge(most_freq_product_name.rename(columns={'product_name':'most_freq_product_name'}), how='left', on='product_code')
df_clean['product_name'] = df_clean['most_freq_product_name']
df_clean = df_clean.drop(columns='most_freq_product_name')
# menghapus outlier
from scipy import stats
df_clean = df_clean[(np.abs(stats.zscore(df_clean[['quantity','amount']]))<3).all(axis=1)]
df_clean = df_clean.reset_index(drop=True)
df_clean

,order_id,product_code,product_name,quantity,order_date,price,customer_id,date,amount
0,493414,21844,red retrospot mug,36.0,2010-01-04 10:28:00,2.55,14590.0,2010-01-04,91.8
1,493414,21533,retro spot large milk jug,12.0,2010-01-04 10:28:00,4.25,14590.0,2010-01-04,51.0
2,493414,37508,new england ceramic cake server,2.0,2010-01-04 10:28:00,2.55,14590.0,2010-01-04,5.1
3,493414,35001G,hand open shape gold,2.0,2010-01-04 10:28:00,4.25,14590.0,2010-01-04,8.5
4,493414,21527,red retrospot traditional teapot,12.0,2010-01-04 10:28:00,6.95,14590.0,2010-01-04,83.4
...,...,...,...,...,...,...,...,...,...
206426,522423,21733,red hanging heart t-light holder,12.0,2010-09-14 12:55:00,2.95,13880.0,2010-09-14,35.4
206427,522423,20969,red floral feltcraft shoulder bag,4.0,2010-09-14 12:55:00,3.75,13880.0,2010-09-14,15.0
206428,522426,22631,circus parade lunch box,2.0,2010-09-14 13:16:00,1.95,17861.0,2010-09-14,3.9
206429,522426,22629,spaceboy lunch box,2.0,2010-09-14 13:16:00,1.95,17861.0,2010-09-14,3.9


In [8]:
#membuat dataframe basket

basket = pd.pivot_table(df_clean, index='order_id', columns='product_name', values='product_code', aggfunc='nunique', fill_value=0)
basket

product_name,10 colour spaceboy pen,12 ass zinc christmas decorations,12 coloured party balloons,12 daisy pegs in wood box,12 egg house painted wood,12 ivory rose peg place settings,12 mini toadstool pegs,12 pencil small tube woodland,12 pencils small tube posy,12 pencils small tube red spotty,...,zinc heart lattice charger large,zinc heart lattice charger small,zinc heart lattice double planter,zinc heart lattice planter bowl,zinc heart lattice t-light holder,zinc heart lattice tray oval,zinc metal heart decoration,zinc police box lantern,zinc top 2 door wooden shelf,zinc willie winkie candle stick
order_id,,,,,,,,,,,,,,,,,,,,,
493414,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
493427,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
493428,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
493432,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
493433,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522418,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
522419,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
522422,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
basket.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9842 entries, 493414 to 522426
Columns: 3555 entries, 10 colour spaceboy pen to zinc willie winkie  candle stick
dtypes: int64(3555)
memory usage: 267.0+ MB


In [11]:
#encode dataframe basket dengan nilai True untuk semua nilai di atas 0 dan False untuk semua nilai 0
def encode(x) :
  if x==0:
    return False
  if x>0:
    return True
basket_encode = basket.applymap(encode)
basket_encode

/tmp/ipykernel_1609/3947176480.py:7: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  basket_encode = basket.applymap(encode)


product_name,10 colour spaceboy pen,12 ass zinc christmas decorations,12 coloured party balloons,12 daisy pegs in wood box,12 egg house painted wood,12 ivory rose peg place settings,12 mini toadstool pegs,12 pencil small tube woodland,12 pencils small tube posy,12 pencils small tube red spotty,...,zinc heart lattice charger large,zinc heart lattice charger small,zinc heart lattice double planter,zinc heart lattice planter bowl,zinc heart lattice t-light holder,zinc heart lattice tray oval,zinc metal heart decoration,zinc police box lantern,zinc top 2 door wooden shelf,zinc willie winkie candle stick
order_id,,,,,,,,,,,,,,,,,,,,,
493414,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
493427,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
493428,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
493432,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
493433,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522418,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
522419,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
522422,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [12]:
#ambil transaksi dengan banyaknya produk unik lebih dari 1 saja

basket_filter = basket_encode[(basket_encode>0).sum(axis=1)>1]
basket_filter

product_name,10 colour spaceboy pen,12 ass zinc christmas decorations,12 coloured party balloons,12 daisy pegs in wood box,12 egg house painted wood,12 ivory rose peg place settings,12 mini toadstool pegs,12 pencil small tube woodland,12 pencils small tube posy,12 pencils small tube red spotty,...,zinc heart lattice charger large,zinc heart lattice charger small,zinc heart lattice double planter,zinc heart lattice planter bowl,zinc heart lattice t-light holder,zinc heart lattice tray oval,zinc metal heart decoration,zinc police box lantern,zinc top 2 door wooden shelf,zinc willie winkie candle stick
order_id,,,,,,,,,,,,,,,,,,,,,
493414,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
493427,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
493428,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
493432,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
493433,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522418,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
522419,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
522422,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


# **Mengaplikasikan Apriori**

**Buat list frequent itemset (kumpulan produk yang sering dibeli)**

In [13]:
# pip install mlxtend
from mlxtend.frequent_patterns import apriori

frequent_itemset = apriori(basket_filter, min_support=.01, use_colnames=True).sort_values('support', ascending=False).reset_index(drop=True)
frequent_itemset['product_cnt'] = frequent_itemset['itemsets'].apply(lambda x: len(x))
frequent_itemset

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,support,itemsets,product_cnt
0,0.194512,(white hanging heart t-light holder),1
1,0.102821,(jumbo bag red retrospot),1
2,0.095327,(regency cakestand 3 tier),1
3,0.090478,(lunch bag red spotty),1
4,0.087613,(pack of 72 retro spot cake cases),1
...,...,...,...
1330,0.010029,"(jumbo shopper vintage red paisley, jumbo bag ...",3
1331,0.010029,"(key fob , back door , key fob , front door ,...",3
1332,0.010029,"(jumbo storage bag suki, lunch bag pink retros...",3
1333,0.010029,(36 pencils tube skulls),1


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Hitung nilai support, confidence, dan lift dari setiap pasangan produk yang mungkin**

In [14]:
from mlxtend.frequent_patterns import association_rules

product_association = association_rules(frequent_itemset, metric='confidence', min_threshold=.7).sort_values(['support','confidence'], ascending=[False,False]).reset_index(drop=True)
product_association

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(red hanging heart t-light holder),(white hanging heart t-light holder),0.051356,0.194512,0.039564,0.770386,3.960615,1.0,0.029574,3.508014,0.787981,0.191774,0.714938,0.486893
1,(sweetheart ceramic trinket box),(strawberry ceramic trinket box),0.052127,0.084637,0.039233,0.752643,8.892552,1.0,0.034821,3.700568,0.936356,0.402260,0.729771,0.608092
2,(wooden picture frame white finish),(wooden frame antique white),0.044743,0.065352,0.032510,0.726601,11.118343,1.0,0.029586,3.418624,0.952685,0.419034,0.707485,0.612036
3,(toilet metal sign),(bathroom metal sign),0.035596,0.049592,0.028433,0.798762,16.106584,1.0,0.026668,4.722795,0.972532,0.500971,0.788261,0.686047
4,(red retrospot sugar jam bowl),(red retrospot small milk jug),0.028212,0.047829,0.021159,0.750000,15.680876,1.0,0.019810,3.808684,0.963408,0.385542,0.737442,0.596198
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151,"(strawberry charlotte bag, woodland charlotte ...",(red spotty charlotte bag),0.013004,0.044082,0.010029,0.771186,17.494364,1.0,0.009455,4.177716,0.955261,0.213115,0.760635,0.499343
152,"(key fob , back door , key fob , front door )","(key fob , garage design)",0.013555,0.022041,0.010029,0.739837,33.566423,1.0,0.009730,3.759030,0.983540,0.392241,0.733974,0.597419
153,"(lunch bag spaceboy design, lunch bag woodland...",(lunch bag suki design),0.013886,0.070972,0.010029,0.722222,10.176156,1.0,0.009043,3.344501,0.914429,0.134021,0.701002,0.431763
154,"(lunch bag spaceboy design, lunch bag woodland...",(lunch bag pink retrospot),0.014216,0.072184,0.010029,0.705426,9.772578,1.0,0.009002,3.149690,0.910619,0.131313,0.682508,0.422179


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
